# Qwen3.5 DataMate — Kaggle T4 ×2
Bật Internet, GPU T4 ×2 và gắn dataset. Chạy 5 câu trước.


In [ ]:
REPO_URL = "https://github.com/KhoaNotFound/ZAIC2026-BackpropEverytTime.git"  # sửa URL
DATA_DIR = "/kaggle/input/<dataset>/train"  # sửa đường dẫn
OUTPUT = "/kaggle/working/smoke/predictions.jsonl"
LIMIT = 5
RESUME = False


In [ ]:
import subprocess
from pathlib import Path
repo = Path("/kaggle/working/zAIC")
if not repo.exists():
    subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True)
%cd /kaggle/working/zAIC
%pip install -r requirements.txt


Nếu pip yêu cầu restart kernel, restart rồi chạy lại cell cấu hình trước khi tiếp tục.


In [ ]:
import torch
print("PyTorch:", torch.__version__, "CUDA:", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(i, props.name, round(props.total_memory / 2**30, 2), "GiB")


In [ ]:
import subprocess, sys, os
os.chdir("/kaggle/working/zAIC")
from pathlib import Path
assert (Path(DATA_DIR) / "questions.jsonl").is_file(), "Sửa DATA_DIR"
base = [sys.executable, "inference.py", "--data-dir", DATA_DIR, "--limit", str(LIMIT), "--output", OUTPUT]
subprocess.run(base + ["--dry-run"], check=True)


In [ ]:
command = base + ["--model", "Qwen/Qwen3.5-9B", "--quantization", "4bit", "--device-map", "split"]
if RESUME:
    command += ["--resume"]
subprocess.run(command, check=True)


In [ ]:
if (Path(DATA_DIR) / "answers.jsonl").is_file():
    subprocess.run([sys.executable, "evaluate.py", "--predictions", OUTPUT,
                    "--answers", str(Path(DATA_DIR)/"answers.jsonl"),
                    "--questions", str(Path(DATA_DIR)/"questions.jsonl")], check=True)
else:
    print("Không có nhãn; chỉ lưu dự đoán.")


In [ ]:
summary = Path(OUTPUT).with_suffix(".summary.json")
if summary.exists():
    print(summary.read_text())
print("Artifacts:", [p.name for p in Path(OUTPUT).parent.glob("predictions.*")])


Muốn chạy 20 câu: đổi LIMIT và dùng OUTPUT mới. Chạy tiếp cùng đợt: giữ cấu hình, đặt RESUME=True. Save Version để giữ output. Không đưa answers.jsonl vào thư mục doanh nghiệp.
